# Gradient Boosting — Performance with Discipline (and Leakage Avoidance)

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb13_gradient_boosting.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how gradient boosting reduces **bias** by training trees sequentially on the residuals of the previous trees, on both classification and regression spines.
2. Fit `GradientBoostingClassifier` and `GradientBoostingRegressor`; demonstrate the CV-score lift over the random forest from nb12 and the **Week-2 reference**.
3. Diagnose the `learning_rate × n_estimators` trade-off — why "lots of trees, slow learning rate" beats "few trees, fast learning rate" at the same total fit budget.
4. Use **`staged_predict`** to plot train vs validation loss per iteration and identify the early-stopping point.
5. Recognize and prevent **boosting's amplification of leaky features** — the failure mode that nb09's leakage case studies set up.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one per spine. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Random forests reduce **variance** by averaging many trees. Gradient boosting reduces **bias** by stacking trees sequentially — each new tree fits the *residuals* (regression) or *misclassifications* (classification) of the running ensemble. The two strategies attack different parts of the bias/variance decomposition; combining their lessons in nb14's selection ceremony usually produces the strongest model on tabular data.

For the **State Health Department's** screening pipeline, the case for gradient boosting is the same as for the random forest — but with one extra ask: *"can we squeeze the last point of ROC-AUC out of the model?"* That last point matters when the false-negative cost is a missed cancer diagnosis. Boosting often delivers it.

For **HomeValue Analytics'** price-prediction model, the case is sharper. The random forest in nb12 already beat the OLS reference by ~20 R² points on California Housing; gradient boosting typically lifts that by another 3–5 points, which translates to ~USD 5–8K reduction in prediction RMSE — meaningful when the median home value being predicted is around USD 200K.

The cost is real: gradient boosting is sequential (cannot fully parallelize), more sensitive to hyperparameters than random forests, and prone to overfitting if you let it run too many iterations. All three risks have specific mitigations covered in this notebook: tune `learning_rate × n_estimators` together, use `staged_predict` to identify the early-stopping point, and **never let gradient boosting see a leaky feature** — boosting will amplify the leak more aggressively than any other algorithm in this course.

> **A question that often comes up here:** *"if boosting is better than forests on tabular data, why not skip forests entirely?"* Three answers. First, forests give you OOB scoring and parallel fits — properties boosting lacks. Second, the four-method importance heatmap from nb12 transfers cleanly to boosting, but the *interpretation* of MDI is harder for boosting (sequential trees mean the importance accumulates differently). Third, the right answer is not "always boost" but "compare both under nb14's CI-overlap discipline and pick the simpler model when CIs overlap." Today's job is to add gradient boosting to the candidate roster — not to crown it.

---

## 1. Setup — Imports, References, Helpers

Same toolkit as nb12: five plot helpers (`plot_train_val_curve`, `plot_predicted_vs_actual`, `plot_cv_ci`, `plot_importance_bars`, `plot_importance_heatmap`) plus the two Week-2 reference pipelines. `staged_predict` is built into sklearn's `GradientBoostingClassifier` / `GradientBoostingRegressor` — no extra import needed.

> 💡 **Gemini Prompt:** "Set up imports for sklearn GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, log_loss, mean_squared_error, train_test_split, cross_val_score, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the Week-2 baseline pipelines. Define helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf`, `reference_reg` defined
> - [ ] All five plot helpers callable
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, Week-2 references, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train', linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s', label='CV ± SD',
                linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual', color=REG_COLOR):
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo, hi = float(min(np.min(y_true), np.min(y_pred))), float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5):
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    ax.errorbar(df['mean'], df['name'], xerr=df['half_w'],
                fmt='o', capsize=6, linewidth=2, color=color, markersize=10)
    for _, r in df.iterrows():
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                r['name'], f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Week-2 references: reference_clf, reference_reg")
print(f"✓ Helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci")


---

## 2. Boosting vs Bagging — Sequential vs Parallel

The schematic below contrasts the two ensemble strategies:

- **Bagging (random forest):** Fit `B` trees in parallel on bootstrap samples. Average (or vote) the predictions. Each tree is full-strength; the ensemble's job is to cancel variance.
- **Boosting (gradient boosting):** Fit tree 1 on the data. Compute residuals. Fit tree 2 on the residuals. Add tree 2's prediction (scaled by `learning_rate`) to tree 1's. Compute new residuals. Repeat. Each tree is **deliberately weak** (shallow, like depth 3); the ensemble's job is to incrementally reduce bias.

In [ ]:
# Schematic: bagging vs boosting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bagging — parallel fan
ax = axes[0]
for i in range(5):
    ax.annotate('', xy=(0.55, 0.5), xytext=(0.15, 0.85 - i*0.15),
                arrowprops=dict(arrowstyle='->', color=CLF_COLOR, lw=2))
    ax.text(0.10, 0.85 - i*0.15, f'Tree {i+1}', ha='right', va='center', fontsize=11)
ax.text(0.55, 0.5, 'Average\n(or vote)', ha='center', va='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.text(0.85, 0.5, 'Prediction', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=CLF_COLOR, alpha=0.3))
ax.annotate('', xy=(0.95, 0.5), xytext=(0.65, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.set_title('Bagging (Random Forest) — parallel, independent trees', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# Boosting — sequential chain
ax = axes[1]
for i in range(5):
    x_pos = 0.10 + i*0.18
    ax.text(x_pos, 0.5, f'T{i+1}', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=REG_COLOR, alpha=0.3))
    if i < 4:
        ax.annotate('', xy=(x_pos + 0.13, 0.5), xytext=(x_pos + 0.05, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(x_pos, 0.30, 'fits\nresiduals\nof T1..T'+str(i) if i > 0 else 'fits\nfull data',
            ha='center', va='center', fontsize=8, color=GREY)
ax.text(0.5, 0.78, 'Σ scaled by learning_rate',
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.set_title('Boosting (Gradient Boosting) — sequential, each tree fixes the previous error',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

plt.tight_layout()
plt.show()


**Reading the output:**

The two diagrams capture the structural difference. Bagging is **parallel and order-independent** — train tree 1 and tree 5 simultaneously and average them at the end. Boosting is **sequential and order-dependent** — tree 5 is fit on residuals that tree 1 → tree 4 produced, so the order matters and the trees cannot be trained in parallel.

The order dependence is what makes boosting slower per training run than a forest of the same size — you cannot parallelize across trees. It is also what makes boosting more accurate on tabular data, because each new tree's job is *specifically defined* (fix what is currently broken) rather than just "be a different tree."

**Key takeaway:** Bagging cancels variance by averaging independent trees; boosting reduces bias by stacking trees that each correct the previous error. nb14 will compare both under identical CV folds — neither is universally better.

---

## 3. Load Both Datasets

Same 60/20/20 locking discipline as nb11 and nb12. Test sets remain sealed until nb14.

In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# 60/20/20 split — same partition as nb11 / nb12 (locked by RANDOM_SEED).
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_clf_3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_reg_3 = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification — Train: {len(X_train_clf)} | Val: {len(X_val_clf)} | Test: {len(X_test_clf)} (LOCKED)')
print(f'Regression     — Train: {len(X_train_reg)} | Val: {len(X_val_reg)} | Test: {len(X_test_reg)} (LOCKED)')


---

## 4. Baseline Gradient Boosting — Default Hyperparameters on Both Spines

The sklearn defaults for `GradientBoostingClassifier` / `GradientBoostingRegressor` are:

- `n_estimators=100` — 100 boosting iterations
- `learning_rate=0.1` — each tree's contribution is shrunk by 0.1
- `max_depth=3` — each tree is shallow (the "weak learner" idea)
- `loss='log_loss'` (clf) / `'squared_error'` (reg)

The defaults are surprisingly competitive. We compare them against the random forest from nb12 and against the Week-2 reference, on identical CV folds.

In [ ]:
gbm_clf = GradientBoostingClassifier(random_state=RANDOM_SEED)
gbm_reg = GradientBoostingRegressor(random_state=RANDOM_SEED)
forest_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
forest_reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)

clf_scores = {
    'Week-2 reference (LogReg)': cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100)':       cross_val_score(forest_clf,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Gradient Boosting (def)':   cross_val_score(gbm_clf,       X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'Week-2 reference (OLS)':    cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (100)':       cross_val_score(forest_reg,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Gradient Boosting (def)':   cross_val_score(gbm_reg,       X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

print("=== CLASSIFICATION ===")
for name, s in clf_scores.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")
print()
print("=== REGRESSION ===")
for name, s in reg_scores.items():
    print(f"  {name}:  mean = {s.mean():.4f}  SD = {s.std(ddof=1):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_cv_ci(clf_scores, 'ROC-AUC', 'Classification — GBM vs RF vs Reference', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_scores, 'R²',      'Regression — GBM vs RF vs Reference', axes[1], color=REG_COLOR)
fig.suptitle('Default GBM is already competitive — CI-clear lift on regression, ties on classification',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On classification, default GBM lands close to the random forest and to the LogReg reference — the three CIs overlap heavily on Wisconsin breast cancer because the dataset is small and mostly linearly separable. By the CI-overlap rule, no model has earned displacement.

On regression, **default GBM does not actually beat default Random Forest** — the forest's CV R² (~0.80) edges past the default GBM (~0.78) by a CI-clear margin. This is a useful result: it shows that **algorithm choice is not the only lever**. Default `GradientBoostingRegressor` uses `max_depth=3` (shallow weak learners), while default `RandomForestRegressor` lets each tree grow deep. On California Housing the deeper RF trees capture more structure than the shallow boosted ensemble. Section 8 will fix this by **tuning GBM's depth**, at which point GBM pulls ahead.

Default GBM is decisively above OLS (~0.60), so the candidate has earned its place in the nb14 roster — but the lift over RF requires tuning, not just defaults.

**Key takeaway:** Default GBM is competitive on classification and beats OLS by a CI-clear margin on regression, but does not automatically beat Random Forest. Tuning depth + n_estimators + learning_rate together is what unlocks GBM's regression edge — Sections 5–8 walk that tuning explicitly.

---

## 5. Learning Rate Trade-off — Slow Learning Beats Fast Learning

`learning_rate` (also called *shrinkage*) scales each tree's contribution to the running ensemble. With `learning_rate=0.1` and 100 trees, the final prediction is `0.1 × (T1 + T2 + ... + T100)`. With `learning_rate=0.5` and 100 trees, each tree contributes 5× more — but the ensemble is more prone to overshoot the optimum.

The key trade-off: **lower `learning_rate` requires more `n_estimators`**, but typically achieves a lower asymptote on validation loss. The sweep below holds `n_estimators=100` and varies `learning_rate ∈ [0.01, 0.05, 0.1, 0.2, 0.5]`.

In [ ]:
lr_grid = [0.01, 0.05, 0.1, 0.2, 0.5]

clf_lr_means, clf_lr_sds = [], []
for lr in lr_grid:
    m = GradientBoostingClassifier(learning_rate=lr, random_state=RANDOM_SEED)
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
    clf_lr_means.append(s.mean()); clf_lr_sds.append(s.std(ddof=1))

reg_lr_means, reg_lr_sds = [], []
for lr in lr_grid:
    m = GradientBoostingRegressor(learning_rate=lr, random_state=RANDOM_SEED)
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_lr_means.append(s.mean()); reg_lr_sds.append(s.std(ddof=1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.errorbar(lr_grid, clf_lr_means, yerr=clf_lr_sds, marker='o', linewidth=2,
            capsize=5, color=CLF_COLOR, label='3-fold CV ROC-AUC')
ax.set_xscale('log'); ax.set_xlabel('learning_rate (log scale)')
ax.set_ylabel('CV ROC-AUC')
ax.set_title('Classification — learning rate vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.errorbar(lr_grid, reg_lr_means, yerr=reg_lr_sds, marker='o', linewidth=2,
            capsize=5, color=REG_COLOR, label='3-fold CV R²')
ax.set_xscale('log'); ax.set_xlabel('learning_rate (log scale)')
ax.set_ylabel('CV R²')
ax.set_title('Regression — learning rate vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Learning rate sweet spot — too low (underfits at fixed n_estimators) or too high (overshoots)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

Both spines show a U-shaped or asymmetric pattern. At `learning_rate=0.01` with only 100 trees, the ensemble has not had enough iterations to fit — each tree contributes only 1% of its prediction, so 100 trees together only deliver the equivalent of one full tree's worth of signal. At `learning_rate=0.5`, the ensemble overshoots — each tree's contribution is large enough that subsequent trees spend their capacity correcting overshoots from the previous trees.

The sweet spot for fixed `n_estimators=100` is usually around 0.05–0.1. To use a smaller `learning_rate` (e.g., 0.01), you need to compensate with **more trees** — Section 6's grid pairs the two together.

**Key takeaway:** `learning_rate` and `n_estimators` are coupled — never tune them independently. Section 6's joint sweep is the right way.

---

## 6. n_estimators × learning_rate — Joint Tuning Grid

The 3×3 heatmap below pairs `n_estimators ∈ {50, 100, 200}` with `learning_rate ∈ {0.05, 0.1, 0.2}`. The pattern to look for: at low `learning_rate`, you need more `n_estimators` to reach the asymptote; at high `learning_rate`, more trees buy diminishing or negative returns (overshoot accumulates).

In [ ]:
n_grid = [50, 100, 200]
lr_grid_joint = [0.05, 0.1, 0.2]

clf_mat = np.zeros((len(n_grid), len(lr_grid_joint)))
reg_mat = np.zeros((len(n_grid), len(lr_grid_joint)))

for i, n in enumerate(n_grid):
    for j, lr in enumerate(lr_grid_joint):
        m_c = GradientBoostingClassifier(n_estimators=n, learning_rate=lr, random_state=RANDOM_SEED)
        s_c = cross_val_score(m_c, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
        clf_mat[i, j] = s_c.mean()

        m_r = GradientBoostingRegressor(n_estimators=n, learning_rate=lr, random_state=RANDOM_SEED)
        s_r = cross_val_score(m_r, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
        reg_mat[i, j] = s_r.mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, mat, title, cbar_label, color_map in [
    (axes[0], clf_mat, 'Classification — CV ROC-AUC', 'CV ROC-AUC', 'YlOrRd'),
    (axes[1], reg_mat, 'Regression — CV R²',           'CV R²',      'YlOrRd'),
]:
    im = ax.imshow(mat, cmap=color_map, aspect='auto')
    ax.set_xticks(range(len(lr_grid_joint))); ax.set_xticklabels([f'lr={lr}' for lr in lr_grid_joint])
    ax.set_yticks(range(len(n_grid)));        ax.set_yticklabels([f'n={n}' for n in n_grid])
    ax.set_xlabel('learning_rate'); ax.set_ylabel('n_estimators')
    ax.set_title(title, fontsize=12, fontweight='bold')
    best = np.unravel_index(mat.argmax(), mat.shape)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            mark = ' ★' if (i, j) == best else ''
            ax.text(j, i, f'{mat[i, j]:.4f}{mark}', ha='center', va='center',
                    color='black', fontweight='bold' if (i,j)==best else 'normal')
    plt.colorbar(im, ax=ax, label=cbar_label, shrink=0.8)

fig.suptitle('Joint tuning — small lr needs more n_estimators; the diagonal is the sweet spot',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The diagonal pattern — *small lr × large n*, *large lr × small n* — is visible on both heatmaps. At `lr=0.05` the best `n_estimators` is 200 (more trees needed to reach the asymptote); at `lr=0.2` the best `n_estimators` shifts down because additional trees overshoot.

For both spines, the global best is usually somewhere along the diagonal — `(lr=0.05, n=200)` or `(lr=0.1, n=100)` — with CV scores within one SD of each other. The one-SE rule typically picks `(lr=0.1, n=100)` because it is faster to fit at no statistical cost.

> **A question that often comes up here:** *"if more trees with slower learning is more accurate, why not run lr=0.001 and n=10,000?"* Compute. At 10,000 trees, GBM fitting time grows linearly while the marginal accuracy gain shrinks. Past `n_estimators=500` or so, the curve is flat enough that the extra trees buy minutes of fit time and almost nothing in CV score. Section 7's early-stopping diagnostic shows this directly.

**Key takeaway:** Tune `learning_rate × n_estimators` together. The diagonal is the sweet spot; the off-diagonal corners are wasteful.

---

## 7. Overfitting in Boosting — `staged_predict` and Early Stopping

Unlike random forests (where more trees never hurt CV score because they only smooth the average), gradient boosting **can overfit** if you let it run too many iterations. The training loss keeps falling toward zero; the validation loss eventually turns up.

`staged_predict` is the right diagnostic. Fit one GBM with a large `n_estimators` (say 500), then iterate over the staged predictions to compute val loss at every iteration — without refitting. The plot below traces train vs val loss for both spines and marks the optimal stopping point as a vertical dashed line.

In [ ]:
# Use the canonical validation set for the staged-predict held-out curve.
# X_val_* was set aside in Section 3 specifically for one-shot held-out checks like this one.

# Classification: log-loss
gbm_c = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                   random_state=RANDOM_SEED).fit(X_train_clf, y_train_clf)
train_loss_c = []
val_loss_c   = []
for proba_train, proba_val in zip(gbm_c.staged_predict_proba(X_train_clf),
                                   gbm_c.staged_predict_proba(X_val_clf)):
    train_loss_c.append(log_loss(y_train_clf, proba_train))
    val_loss_c.append(log_loss(y_val_clf, proba_val))
opt_n_c = int(np.argmin(val_loss_c)) + 1

# Regression: MSE
gbm_r = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                  random_state=RANDOM_SEED).fit(X_train_reg, y_train_reg)
train_loss_r = []
val_loss_r   = []
for pred_train, pred_val in zip(gbm_r.staged_predict(X_train_reg),
                                 gbm_r.staged_predict(X_val_reg)):
    train_loss_r.append(mean_squared_error(y_train_reg, pred_train))
    val_loss_r.append(mean_squared_error(y_val_reg, pred_val))
opt_n_r = int(np.argmin(val_loss_r)) + 1

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(range(1, 301), train_loss_c, label='Train log-loss', color=GREY, linewidth=2)
ax.plot(range(1, 301), val_loss_c,   label='Val log-loss',   color=CLF_COLOR, linewidth=2)
ax.axvline(opt_n_c, color='red', linestyle='--', label=f'Early stop @ n={opt_n_c}')
ax.set_xlabel('n_estimators'); ax.set_ylabel('log-loss')
ax.set_title('Classification — staged loss curves', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, 301), train_loss_r, label='Train MSE', color=GREY, linewidth=2)
ax.plot(range(1, 301), val_loss_r,   label='Val MSE',   color=REG_COLOR, linewidth=2)
ax.axvline(opt_n_r, color='red', linestyle='--', label=f'Early stop @ n={opt_n_r}')
ax.set_xlabel('n_estimators'); ax.set_ylabel('MSE')
ax.set_title('Regression — staged loss curves', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Early stopping — train loss → 0; val loss eventually turns up',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 Classification early-stop n_estimators: {opt_n_c}")
print(f"💡 Regression early-stop n_estimators:     {opt_n_r}")


**Reading the output:**

Train loss falls monotonically toward zero on both spines — the ensemble is incrementally fitting harder and harder corners of the training data. Val loss falls quickly at first, then bottoms out, then begins to rise as the additional trees fit noise that does not generalize. The vertical red dashed line marks the iteration where val loss is minimized — the **early-stopping point**.

For the M3 milestone with gradient boosting, the right protocol is:
1. Use `staged_predict` on a held-out validation split to find the optimal `n_estimators`.
2. Refit on the full training set with that `n_estimators` (no validation set held out).
3. Report CV scores using the chosen `n_estimators`.

> **A question that often comes up here:** *"why doesn't sklearn just have a `n_iter_no_change` parameter?"* It does — `GradientBoostingClassifier(n_iter_no_change=10, validation_fraction=0.1)` automates the early-stopping protocol. Use the manual `staged_predict` approach when you want to *see* the curves; use `n_iter_no_change` in production when you trust the protocol.

**Key takeaway:** Boosting's overfitting is a real risk, but it is also easy to diagnose — `staged_predict` gives you the curve at no extra fit cost. Always plot it before declaring a `n_estimators` value for the M3 model.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingClassifier` on Wisconsin breast cancer using a 3×3 grid.

**Instructions:**
1. Sweep `n_estimators ∈ [50, 100, 200]` × `learning_rate ∈ [0.05, 0.1, 0.2]`.
2. For each combination, compute 3-fold CV ROC-AUC mean and SD using `cv_clf_3`.
3. Render as a heatmap with mean values annotated.
4. Apply the one-SE rule.
5. Write 3 short findings on whether the diagonal pattern from Section 6 confirmed itself.

---

> 💡 **Gemini Prompt:** "Grid-search GradientBoostingClassifier(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using 3-fold CV ROC-AUC. Build a 3×3 heatmap with means annotated; star the best cell; apply the one-SE rule."


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune GradientBoostingClassifier over (n_estimators, learning_rate); apply one-SE rule.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingRegressor` on California Housing.

**Instructions:**
1. Same 3×3 grid as Exercise 1 (`n ∈ [50, 100, 200]` × `lr ∈ [0.05, 0.1, 0.2]`).
2. 3-fold CV R² with `cv_reg_3`.
3. Heatmap with mean annotations and starred best cell.
4. Convert best CV-RMSE to USD; apply the one-SE rule.

---

> 💡 **Gemini Prompt:** "Grid-search GradientBoostingRegressor(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using 3-fold CV R². Heatmap of CV means; report best CV-RMSE in USD; apply the one-SE rule."


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune GradientBoostingRegressor; report best CV-RMSE in USD.


## 8. Final Comparison — All Models on Both Spines

The closing comparison is the direct setup for nb14's selection ceremony. Five models per spine: Week-2 reference, Decision Tree, Random Forest, default GBM, and the tuned GBM from the exercises above. CV-CI dot plot per spine.

In [ ]:
# Build candidate sets per spine
clf_compare = {
    'Week-2 reference (LogReg)': cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Decision Tree (depth=3)':   cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100)':       cross_val_score(RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'GBM (default)':             cross_val_score(GradientBoostingClassifier(random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'GBM (tuned: lr=0.05, n=200, d=3)': cross_val_score(GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}

reg_compare = {
    'Week-2 reference (OLS)':       cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Decision Tree (depth=10)':     cross_val_score(DecisionTreeRegressor(max_depth=10, random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (100)':          cross_val_score(RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'GBM (default, depth=3)':       cross_val_score(GradientBoostingRegressor(random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'GBM (tuned: lr=0.1, n=200, d=5)': cross_val_score(GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

print("=== CLASSIFICATION ===")
for name, s in clf_compare.items():
    print(f"  {name:>32s}:  {s.mean():.4f} ± {s.std(ddof=1):.4f}")
print()
print("=== REGRESSION ===")
for name, s in reg_compare.items():
    print(f"  {name:>32s}:  {s.mean():.4f} ± {s.std(ddof=1):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cv_ci(clf_compare, 'ROC-AUC', 'Classification — full candidate field', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_compare, 'R²',      'Regression — full candidate field',     axes[1], color=REG_COLOR)
fig.suptitle('Five candidates per spine — direct setup for nb14\'s selection ceremony',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On classification, all five candidates' CIs overlap with the LogReg reference. By the CI-overlap rule, no model has earned displacement on this dataset — even the tuned GBM is statistically tied with `LogReg(C=1.0)`. nb14's ceremony will adjudicate this formally; the answer here is that small, mostly-linearly-separable datasets are hard to beat with non-linear models.

On regression, the picture is sharper. OLS is the lowest bar (R² ~0.60). Single Tree lifts above it by ~7 R² points. Random Forest lifts again by ~13 R² points (R² ~0.80). Default GBM (`max_depth=3`) lands slightly *below* RF — shallow boosted trees underfit California Housing. **The tuned GBM with `max_depth=5` (R² ~0.82) is the CI-clear winner**, beating RF by ~2 R² points and OLS by ~22 R² points. CV-RMSE in USD: tuned GBM ~USD 41K vs OLS ~USD 75K — a USD 34K reduction in average prediction error.

The lesson is mechanical: **GBM's edge over RF on regression depends on tuning**, especially `max_depth`. Default GBM with shallow trees is a weaker version of RF; tuned GBM with `max_depth=5` and the right learning rate is materially better. nb14's ceremony will see both default and tuned versions in the candidate roster.

**Key takeaway:** Five candidates per spine are now on the table, plus one or two extras (LogReg L1, Lasso) that nb14 will add. The selection ceremony is one notebook away.

---

## 9. Why Leaky Features Dominate the Top of Boosting's Importance Chart

Gradient boosting amplifies leaky features more aggressively than any other algorithm in this course. The reason is mechanical: every boosting iteration fits the *residuals* of the running ensemble. A leaky feature (one that encodes the target) drives the residuals to zero in the first few iterations, so the algorithm picks it again and again. By iteration 50, the model is essentially `y = leaky_feature` plus tiny corrections from real features.

The diagnostic is the same as nb09's:

1. **Drop or null suspect features one at a time** and re-fit. If CV score collapses by more than ~5 points, the dropped feature was carrying signal that should not have been there.
2. **Inspect the top-ranked feature.** If it is a derived field (e.g., `target_mean_by_group`, `forecast_at_t+1`, `customer_lifetime_value` — anything computed *after* the prediction time), it is leaky.
3. **Check the pipeline boundary.** Was the suspect feature engineered using future data, target data, or held-out data? If yes, leak.

For your M3 milestone with gradient boosting, the protocol is: produce the four-method importance heatmap from nb12, look at the top 3 features, and ask of each: *"could this have been computed at prediction time without seeing the answer?"* If the answer is no for any of them, that feature must come out before the model ships.

> **A question that often comes up here:** *"why is the leakage problem worse for boosting than for forests?"* Because forests **average** trees built from independent bootstraps — a leaky feature is picked at the root of every tree but it is not amplified across iterations. Boosting **chains** trees built on the previous trees' residuals — the leaky feature is picked at the root *and* drives the residual structure all the way down. The forest's importance for a leaky feature is high but bounded; boosting's is high and self-reinforcing.

**Key takeaway:** GBM + leaky feature = catastrophic over-fitting that looks like a great model. Always run the four-method importance check on a tuned GBM before declaring the model ready.

---

## 10. Wrap-Up — Key Takeaways

**What landed today:**

1. **Boosting is sequential, not parallel.** Each tree fits the residuals of the running ensemble; the algorithm reduces bias rather than variance.
2. **Tune `learning_rate × n_estimators` together.** The diagonal of the (lr, n) grid is the sweet spot; the corners are wasteful.
3. **`staged_predict` is the right early-stopping diagnostic.** Fit one large GBM, iterate over staged predictions, find the val-loss minimum.
4. **GBM beats RF on regression by a CI-clear margin; ties on classification.** Same pattern as nb12: small, easy datasets favor simpler models.
5. **Boosting amplifies leaky features.** Always run the four-method importance check on a tuned GBM before declaring the model ready.

**Bridge to nb14 — the Selection Ceremony:**

You now have the full candidate roster: linear references (LogReg / OLS), single tree (nb11), random forest (nb12), default GBM, and tuned GBM (today). Add one regularized linear (LogReg L1 / Lasso) per spine and you have five candidates per case for nb14's CI-overlap ceremony.

nb14 does three things you have not seen as a single workflow yet: (a) **lock the comparison protocol before looking at any results** — same CV folds, same primary metric, same secondary metrics; (b) compute Student's *t* 95% CI on the champion's CV scores and write a defensible **champion memo**; (c) open the **locked test set** for the first and only time, compute the test-set point estimate, and pronounce an INSIDE / ABOVE / BELOW verdict against the CV CI.

This is the payoff for ten notebooks of locking discipline. After nb14, the test set goes back into its envelope — never to be reopened — and nb15 takes the committed champion forward into interpretation.

> **A question that often comes up at this point:** *"if my CV-CI says no candidate has earned displacement of the linear reference, what do I do?"* You ship the linear reference. That is the discipline working as intended. Sometimes the data is genuinely simple enough that a tuned LogReg or OLS is the right model; the value of running boosting was not the boosting itself but the **proof that it did not improve things**, which is a defensible thing to put in front of a stakeholder. nb14's memo template includes a "no displacement" template for exactly this case.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (GBM tuning, classification) and Exercise 2 (GBM tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 13 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a tuning heatmap with the chosen combination starred
- [ ] The staged-loss curve renders for both spines
- [ ] All figures render (none broken)

### Next Step:

- **Notebook 14** — Model Selection Protocol + Test-Set Opening Ceremony (Day 14)

---

<center>

**Thank you!**

</center>